# VOICE-CUE sLLM 파인튜닝 — Kaggle (백그라운드 실행)

기획서 4-다② "폐쇄망 구동이 가능한 오픈소스 한국어 특화 sLLM을 LoRA로 학습" 단계를 실행합니다.

**Colab 대신 Kaggle을 쓰는 이유**: Colab은 인터랙티브 세션이 브라우저 연결에 묶여 있어 자꾸
끊깁니다(Pro+의 "백그라운드 실행"도 2026년 들어 불안정하다는 신고가 많습니다). Kaggle은
"Save Version"을 누르면 그 시점 노트북 전체가 **별도 머신에서 완전히 분리되어** 처음부터
끝까지 자동 실행됩니다 — 브라우저를 닫아도, 컴퓨터를 꺼도 계속 돕니다.

**시작 전에 반드시 (우측 패널 Settings)**:
1. **Accelerator** → GPU P100 (또는 GPU T4 x2)
2. **Internet** → On (꺼져 있으면 GitHub 클론·pip install이 실패합니다)

**실행 방법 — 이 노트북은 두 단계로 씁니다.**
1. 먼저 1~5번 셀을 인터랙티브하게(Shift+Enter) 하나씩 실행해 GPU·설치·데이터·사전 점검까지
   문제없는지 확인합니다. 여기까지는 수 분이면 끝나고, 인터랙티브 세션이 끊겨도 상관없습니다.
2. 문제가 없으면 우측 상단 **Save Version → Save & Run All (Commit)** 을 누릅니다. 그 순간부터
   창을 닫아도 6~9번 셀(베이스라인 → 학습 → 평가 → 병합)이 별도 머신에서 끝까지 돌아갑니다.
   진행 상황은 나중에 아무 때나 들어가서 그 버전의 로그로 확인하면 됩니다.

무료 할당량은 주당 약 30 GPU시간, 세션(커밋 실행 포함)당 최대 약 12시간입니다. 3번 셀 기본값
(1.5B/2에폭, 약 2.4시간)은 여유 있게 들어갑니다.

## 1. GPU 확인

None이 나오면 우측 Settings 패널에서 Accelerator를 GPU로 바꾸고 세션을 다시 시작하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import subprocess, sys
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
if not out.strip():
    sys.exit("GPU가 없습니다. 우측 Settings 패널 > Accelerator에서 GPU(P100 또는 T4 x2)를 선택하세요.")
print(out)

## 2. 저장소 클론 + 의존성 설치

Internet이 꺼져 있으면 이 셀에서 git clone부터 실패합니다 — Settings 패널에서 켜져 있는지 먼저 확인하세요.
설치에 3~5분 걸립니다.

In [ ]:
%cd /kaggle/working
!rm -rf voiceq-air3
!git clone --branch feature/finetune --depth 1 https://github.com/MeinBau/voiceq-air3.git
%cd /kaggle/working/voiceq-air3

!pip install -q -r finetune/requirements-train.txt

import importlib
for mod in ("torch", "transformers", "trl", "peft", "bitsandbytes", "datasets", "accelerate"):
    try:
        print(f"{mod:14s} {importlib.import_module(mod).__version__}")
    except Exception as e:
        print(f"{mod:14s} 로드 실패: {e}")

## 3. 학습 설정

소요 시간은 GPU와 실제 스텝 속도에 따라 편차가 큽니다. 아래는 Kaggle T4 실측을 반영한 범위입니다.
**처음에는 1.5B / 1에폭으로 끝까지 완주시키는 것을 권장**합니다 — 파이프라인이 실제로 도는지부터 확인한 뒤 에폭·모델을 올리는 편이 안전합니다.

| 모델 | 4bit 크기 | 1 에폭 | 2 에폭 |
|---|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | ~1.1GB | **1.5~3h** | 3~6h |
| `Qwen/Qwen2.5-3B-Instruct` | ~2.0GB | 3~6h | 6~12h (권장하지 않음) |

기획서 3-다의 "4bit 2~4GB" 목표에 정확히 맞는 것은 3B다. 3B/3에폭(약 7.2시간)도 Kaggle GPU
세션 한도(약 12시간) 안에 들어오므로, 1.5B로 파이프라인을 먼저 완주시킨 뒤 최종 수치는 3B로
다시 커밋 실행하는 순서를 권장한다.

체크포인트·어댑터·병합본은 모두 `/kaggle/working` 아래에 저장되므로, Save & Run All(Commit)이
끝나면 그 버전의 Output 탭에서 자동으로 받을 수 있다(Colab처럼 Drive 마운트가 필요 없다).

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 3B로 바꾸려면: "Qwen/Qwen2.5-3B-Instruct"
EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8      # 실효 배치 = BATCH_SIZE * GRAD_ACCUM = 16
MAX_SEQ_LEN = 2048  # 데이터 p99가 1699라 잘림 없음

import os
OUT = "/kaggle/working/voiceq-air3/finetune/out"
os.makedirs(OUT, exist_ok=True)
print("체크포인트 경로:", OUT)

## 4. 학습 데이터 생성

데이터는 저장소에 커밋되어 있지 않고 시드 고정 생성물입니다(`--seed 20260829` 기본값).
여기서 만든 것과 로컬에서 만든 것이 완전히 같습니다.

In [ ]:
%cd /kaggle/working/voiceq-air3
!python finetune/gen_dataset.py
assert _exit_code == 0, f"데이터 생성 실패 (exit {_exit_code}) — 위 트레이스백을 확인하세요."

## 5. 학습 전 점검

여기까지가 "인터랙티브하게 눈으로 확인하는" 구간입니다. 학습을 몇 시간 돌린 뒤에 데이터가
깨져 있던 걸 발견하면 그 시간을 통째로 버립니다. 스키마·토큰 길이와 평가 지표 계산식이
정상인지 먼저 확인합니다(gold는 100%가 나와야 정상). **여기까지 이상 없으면 Save & Run All로
넘어가도 됩니다.**

In [ ]:
!python finetune/train_lora.py --dry-run --model $MODEL --max-seq-len $MAX_SEQ_LEN
assert _exit_code == 0, f"dry-run 실패 (exit {_exit_code})"
!python tests/test_tiling.py
assert _exit_code == 0, f"타일링 불변식 테스트 실패 (exit {_exit_code})"
!python finetune/evaluate.py --backend gold --split test
assert _exit_code == 0, f"평가 하네스 자기검증 실패 (exit {_exit_code}) — gold인데 100%가 아니면 지표 코드가 깨진 것"

## 6. 베이스라인 측정 (튜닝 전)

여기부터는 시간이 걸리므로 **Save & Run All(Commit)로 백그라운드 실행하는 것을 권장**합니다.

파인튜닝의 효과를 말하려면 비교 대상이 있어야 합니다. **튜닝 전 모델에는 few-shot을 켜서**
측정합니다 — few-shot 없이 형식을 지키게 만드는 것 자체가 파인튜닝의 성과이므로, 양쪽에
똑같이 켜면 그 효과가 측정되지 않습니다.

전체 test 179턴 × 2경로는 시간이 걸리므로 `--limit 40`으로 표본만 봅니다.

In [ ]:
!python finetune/evaluate.py --backend hf --model $MODEL \
    --split test --limit 40 --few-shot \
    --out $OUT/baseline.json
assert _exit_code == 0, f"베이스라인 평가 실패 (exit {_exit_code}) — 위 트레이스백을 확인하세요."

## 7. QLoRA 학습

여기가 가장 오래 걸리는 부분입니다. 10 스텝마다 loss가 찍히고, 에폭마다 체크포인트와 평가
loss가 남습니다.

GPU가 두 장이면(Kaggle T4 x2) 자동으로 데이터 병렬(DDP)로 두 장을 다 씁니다 — 한 장만 쓸
때보다 거의 두 배 빠릅니다. 실효 배치는 GPU 장수와 무관하게 같게 유지됩니다.

`--save-steps` 주기(기본 25스텝)마다 체크포인트가 남습니다. 도중에 죽거나 끊겨도 이 셀을
다시 실행하면 마지막 체크포인트를 찾아 이어서 학습하므로, 진행분이 통째로 날아가지 않습니다.

다만 인터랙티브 세션이 완전히 종료되면 `/kaggle/working` 내용도 같이 사라집니다. 정말
끊김 없이 돌리려면 위 안내대로 **Save & Run All(Commit)**을 쓰세요.

In [ ]:
import torch

# GPU가 여러 장이면 데이터 병렬(DDP)로 전부 쓴다. Kaggle의 T4 x2가 여기 해당하며,
# 거의 장수에 비례해 빨라진다. 한 장뿐이면(Colab T4, Kaggle P100) 그냥 python으로 띄운다.
N_GPU = torch.cuda.device_count()
# 실효 배치(BATCH_SIZE x GRAD_ACCUM x 프로세스 수)를 GPU 장수와 무관하게 고정한다.
ACCUM = max(1, GRAD_ACCUM // max(1, N_GPU))
LAUNCH = (f"accelerate launch --num_processes {N_GPU} --mixed_precision fp16"
          if N_GPU > 1 else "python")
print(f"GPU {N_GPU}장 · 실행 방식: {LAUNCH.split()[0]} · "
      f"실효 배치 {BATCH_SIZE}x{ACCUM}x{max(1, N_GPU)} = {BATCH_SIZE * ACCUM * max(1, N_GPU)}")

!{LAUNCH} finetune/train_lora.py \
    --model $MODEL \
    --out $OUT \
    --epochs $EPOCHS \
    --batch-size $BATCH_SIZE \
    --grad-accum $ACCUM \
    --max-seq-len $MAX_SEQ_LEN
assert _exit_code == 0, (
    f"학습 실패 (exit {_exit_code}) — 위 트레이스백이 진짜 원인입니다. "
    "여기서 멈추므로 아래 평가 셀이 이어서 도는 일은 없습니다."
)

## 8. 튜닝 모델 평가

튜닝 후에는 **few-shot 없이** 측정합니다. 6번 셀의 베이스라인과 비교하면 파인튜닝 효과가 나옵니다.

In [ ]:
!python finetune/evaluate.py --backend hf --model $MODEL \
    --adapter $OUT/adapter \
    --split test --limit 40 \
    --out $OUT/tuned.json
assert _exit_code == 0, f"튜닝 모델 평가 실패 (exit {_exit_code}) — 위 트레이스백을 확인하세요."

In [ ]:
import json

base = json.load(open(f"{OUT}/baseline.json", encoding="utf-8"))
tuned = json.load(open(f"{OUT}/tuned.json", encoding="utf-8"))

ROWS = [
    ("지표② 상황유형 정확도",  "지표②_상위배치정확도", "situation_accuracy_pct", "90% 이상"),
    ("지표② COP 셀 일치율",   "지표②_상위배치정확도", "cop_cell_match_pct",    "90% 이상"),
    ("지표④ 키워드 정확도",    "지표④_일지정확도",     "keyword_accuracy_pct",  "90% 이상"),
    ("지표④ 누락률",          "지표④_일지정확도",     "omission_rate_pct",     "5% 미만"),
    ("지표④ 일지 kind 정확도", "지표④_일지정확도",     "log_kind_accuracy_pct", "-"),
    ("지표④ ROUGE-L",        "지표④_일지정확도",     "rouge_l_pct",           "-"),
    ("JSON 유효율 (FAST)",    "부가",                "fast_json_valid_pct",   "-"),
    ("지표① FAST 지연(초)",   "지표①_표출지연",       "fast_p50_sec",          "5초 이내"),
]

print(f"{'지표':24s} {'튜닝 전':>10s} {'튜닝 후':>10s} {'변화':>10s}   목표")
print("-" * 74)
for label, group, key, target in ROWS:
    b, t = base[group][key], tuned[group][key]
    print(f"{label:24s} {b:10.2f} {t:10.2f} {t - b:+10.2f}   {target}")

## 9. 폐쇄망 서빙용 병합 (선택)

어댑터를 베이스에 합쳐 통짜 가중치로 만듭니다. vLLM/Ollama로 부대 내 서버에 올릴 때 쓰며,
앱에서는 사이드바 공급자를 "로컬 서버"로 바꾸기만 하면 코드 수정 없이 연결됩니다.

병합본은 fp16이라 3B 기준 약 6GB다. Kaggle의 노트북 Output 용량 한도가 있으므로, 병합이
끝나면 `checkpoint-*` 중간 산출물은 지우고 `adapter`·`merged`만 남기는 것을 권장한다.

In [ ]:
!python finetune/train_lora.py --model $MODEL --out $OUT --merge $OUT/merged
assert _exit_code == 0, f"병합 실패 (exit {_exit_code})"
!du -sh $OUT/merged

In [ ]:
# Output 용량을 줄이려면: 중간 체크포인트 삭제 (adapter/merged는 남긴다)
!rm -rf $OUT/checkpoint-*
!du -sh $OUT/*